# Generacion de Secuencias Supervisadas v01

Este notebook transforma el dataset tabular procesado en tensores aptos para una LSTM.


## Definicion baseline

- ventana de entrada: 24 horas
- horizonte: 12 horas
- objetivo: helada a `t+12h`

La muestra supervisada representa una decision de pronostico hecha en el tiempo origen, usando solo historia observable hasta ese instante.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'frost_ema_igp' and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

from src.config import SEQUENCE_LENGTH_HOURS
from src.data.make_dataset import build_and_save_datasets
from src.features.preprocessing import build_target_split_masks, impute_features_past_only, select_feature_columns
from src.features.sequence import create_sequence_split

build_and_save_datasets()
processed_df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'frost_dataset_v01.csv', index_col=0, parse_dates=True)
processed_df['target_timestamp'] = pd.to_datetime(processed_df['target_timestamp'])
feature_columns = select_feature_columns(processed_df.columns)
split_masks = build_target_split_masks(processed_df)

feature_df, _ = impute_features_past_only(processed_df[feature_columns], split_masks['train'])
scaler = StandardScaler().fit(feature_df.loc[split_masks['train']])
scaled_df = pd.DataFrame(scaler.transform(feature_df), index=feature_df.index, columns=feature_columns)


In [ ]:
train_positions = split_masks['train'].to_numpy().nonzero()[0]
train_split = create_sequence_split(
    feature_array=scaled_df.to_numpy(),
    target_array=processed_df['frost_event_t_plus_12h'].to_numpy(),
    origin_timestamps=scaled_df.index,
    target_timestamps=pd.DatetimeIndex(processed_df['target_timestamp']),
    selected_positions=train_positions,
    sequence_length=SEQUENCE_LENGTH_HOURS,
)

train_split.X.shape, train_split.y.shape


## Chequeo metodologico

Antes de entrenar conviene verificar siempre:

- que el tensor tenga la forma esperada;
- que el target siga desbalanceado pero viable;
- que la primera secuencia solo use pasado observable.
